# Extract Data From the Returns SQL Table
1. Create Bronze Schema in Hive Metastore
2. Create External Table

## 1. Create Bronze Schema in Hive Metastore

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS hive_metastore.bronze;

## 2. Create External Table

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    DoubleType,
    StringType,
    TimestampType,
    FloatType,
    DecimalType
)
schema_py = StructType([
    StructField("refund_id", IntegerType(), True),
    StructField("payment_id", IntegerType(), True),
    StructField("refund_timestamp", TimestampType(), True),
    StructField("refund_amount", DecimalType(10, 2), True),
    StructField("refund_reason", StringType(), True)
])

In [0]:
df = spark.createDataFrame([
    (1, 66, '2025-01-10 11:30:00', 85.75, 'Payment Error:Retailer'),  
    (2, 69, '2025-01-03 12:40:15', 120.50, 'Order Cancelled:Customer'),  
    (3, 72, '2025-01-06 14:45:30', 65.00, 'Product Returned:Customer'),  
    (4, 73, '2025-01-07 16:10:45', 210.99, 'Order Cancelled:Customer'),  
    (5, 75, '2025-01-09 18:25:00', 45.20, 'Payment Error:Retailer'),  
    (6, 80, '2025-01-10 09:35:20', 130.15, 'Order Cancelled:Customer'),  
    (7, 83, '2025-01-12 11:20:40', 150.00, 'Product Returned:Customer'),  
    (8, 85, '2025-01-14 13:15:30', 89.99, 'Order Cancelled:Customer'),  
    (9, 89, '2025-01-15 15:00:00', 78.50, 'Payment Error:Retailer'),  
    (10, 91, '2025-01-17 16:45:15', 250.75, 'Product Returned:Customer')
    ], schema = ["refund_id", "payment_id", "refund_timestamp", "refund_amount", "refund_reason"]
)
df.show(5)

In [0]:
from pyspark.sql.functions import col
df = (
    df.withColumn("refund_timestamp", col("refund_timestamp").cast("timestamp"))
      .withColumn("refund_amount", col("refund_amount").cast("decimal(10,2)"))
      .withColumn("refund_id", col("refund_id").cast("integer"))
      .withColumn("payment_id", col("payment_id").cast("integer"))
)
df.show(5)

In [0]:
empty_df = spark.createDataFrame([], schema_py)

In [0]:
empty_df.writeTo("hive_metastore.bronze.py_refunds").createOrReplace()

In [0]:
df.write \
    .mode("append") \
    .saveAsTable("hive_metastore.bronze.py_refunds") 

In [0]:
data = spark.sql("select * from hive_metastore.bronze.py_refunds;")
display(data)